# NullWasted: Software Verification & Testing Suite
## Rekayasa Perangkat Lunak (RPL-2) — Kelompok 16 FTUI

Dokumen ini memuat eksekusi kasus uji terstruktur untuk modul inti backend NullWasted.
Setiap sel secara transparan menampilkan:
- **Injected Input**: Data mentah, parameter, atau HTTP payload yang disuntikkan ke dalam sistem.
- **System Logic**: Logika bisnis atau algoritma rekayasa yang dipicu.
- **Verification / Assertion**: Evaluasi kepatuhan luaran sistem terhadap kriteria akseptansi formal.

Standar Acuan: IEEE SWEBOK v4, ISO/IEC/IEEE 29119, ISO/IEC 25010, PerBPOM No. 16/2019.

### 0. Inisialisasi Environment & Konfigurasi Pengujian

In [1]:
import sys
from pathlib import Path

backend_dir = Path.cwd().resolve()
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

import pytest
from fastapi.testclient import TestClient
from sqlmodel import Session, SQLModel, create_engine
from sqlmodel.pool import StaticPool

from app.main import app
from app.core.database import get_session
from app.services.pricing import calculate_dynamic_pricing
from app.services.guardrail import evaluate_food_safety
from app.services.vector_store import vector_store

print("[INITIALIZATION] Backend modules and services loaded.")
print(f"[INJECTED CONFIG] Root Workspace : {backend_dir}")
print(f"[KNOWLEDGE BASE] Food Archetypes: {len(vector_store.items)} items (PerBPOM No. 16/2019).")
print("[VECTOR ENGINE] Embedding Cache : 3072-dimensional vector space.")
print("[DATABASE STATE] Target Database : In-Memory SQLite Isolation (Zero Persistent Contamination).")

C:\Users\Vincenzo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


[INITIALIZATION] Backend modules and services loaded.
[INJECTED CONFIG] Root Workspace : D:\Downloads\Materi\VSCode\RPL\project\backend
[KNOWLEDGE BASE] Food Archetypes: 71 items (PerBPOM No. 16/2019).
[VECTOR ENGINE] Embedding Cache : 3072-dimensional vector space.
[DATABASE STATE] Target Database : In-Memory SQLite Isolation (Zero Persistent Contamination).


---
### 1. Automated Regression Suite (Pytest Runner)
- **Injected Target**: Berkas uji `tests/test_unit_core.py` (33 kasus uji otomatis komprehensif).
- **System Logic**: Menjalankan seluruh pengujian unit, batas diskon BVA, guardrail BPOM, smart bundling, multi-tenant auth, anti-fraud lintas-gerai, serta pipeline upload & inheritance gambar menu.
- **Assertion**: Nilai exit code wajib 0 (100% / 33 test cases lolos).

In [2]:
import concurrent.futures

print("[RUNNER] Invoking Pytest on test suite: tests/test_unit_core.py\n")

def _run_pytest():
    return pytest.main(["-v", "tests/test_unit_core.py"])

# Run in an isolated worker thread so Pytest has a clean execution context independent of Jupyter's active event loop
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    exit_code = executor.submit(_run_pytest).result()

print(f"\n[SUITE RESULT] Execution finished with exit code: {int(exit_code)} (0 = 100% PASSED)")

[RUNNER] Invoking Pytest on test suite: tests/test_unit_core.py

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\Vincenzo\AppData\Local\Python\pythoncore-3.14-64\python.exe
cachedir: .pytest_cache
rootdir: D:\Downloads\Materi\VSCode\RPL\project\backend
configfile: pytest.ini
plugins: anyio-4.12.1, asyncio-1.4.1
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 33 items

tests/test_unit_core.py::test_pricing_happy_path_pill_1 

PASSED           [  3%]


tests/test_unit_core.py::test_pricing_boundary_critical_pill_3 

PASSED    [  6%]


tests/test_unit_core.py::test_pricing_bounds_and_rounding 

... (59 outputs left out)


tests/test_unit_core.py::test_image_url_lifecycle_inheritance_and_adhoc_none 

PASSED [100%]



============================== warnings summary ===============================
C:\Users\Vincenzo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\_pytest\config\__init__.py:1345
  C:\Users\Vincenzo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\_pytest\config\__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================= 33 passed, 1 warning in 32.60s ========================



[SUITE RESULT] Execution finished with exit code: 0 (0 = 100% PASSED)


---
### 2. Unit Testing: Dynamic Pricing Engine
- **Metode Rekayasa**: Boundary Value Analysis (BVA).
- **System Logic**: Formula time-decay yang memperhitungkan rasio waktu operasional tersisa terhadap jam tutup gerai.
- **Injected Inputs**:
  1. `original_price` (Harga dasar hidangan dalam Rupiah).
  2. `minutes_until_closing` (Sisa menit menuju jam tutup operasional gerai: 240 mnt, 120 mnt, 15 mnt).
  3. `custom_discount_percent` (Diskon manual di luar batas: 85% dan 30%).
- **Assertion**: Diskon wajib terkunci pada interval [50%, 70%] dan harga dibulatkan ke ratusan terdekat.

In [3]:
pricing_scenarios = [
    {
        "case_id": "BVA-01",
        "desc": "Far from closing window (240 min left)",
        "injected_price": 20000,
        "injected_minutes": 240,
        "injected_custom_discount": None,
        "expected_discount": 50,
        "expected_price": 10000,
    },
    {
        "case_id": "BVA-02",
        "desc": "Preset Pill 1 standard window (120 min left)",
        "injected_price": 20000,
        "injected_minutes": 120,
        "injected_custom_discount": None,
        "expected_discount": 55,
        "expected_price": 9000,
    },
    {
        "case_id": "BVA-03",
        "desc": "Preset Pill 3 critical close window (15 min left)",
        "injected_price": 30000,
        "injected_minutes": 15,
        "injected_custom_discount": None,
        "expected_discount": 70,
        "expected_price": 9000,
    },
    {
        "case_id": "BVA-04",
        "desc": "Custom discount exceeds maximum ceiling (85% input)",
        "injected_price": 10000,
        "injected_minutes": 60,
        "injected_custom_discount": 85,
        "expected_discount": 70,
        "expected_price": 3000,
    },
    {
        "case_id": "BVA-05",
        "desc": "Custom discount below minimum floor (30% input)",
        "injected_price": 10000,
        "injected_minutes": 60,
        "injected_custom_discount": 30,
        "expected_discount": 50,
        "expected_price": 5000,
    },
    {
        "case_id": "BVA-06",
        "desc": "Odd nominal rounding verification (Rp16.250 / 160 min)",
        "injected_price": 16250,
        "injected_minutes": 160,
        "injected_custom_discount": None,
        "expected_discount": 50,
        "expected_price": 8100,
    },
]

for sc in pricing_scenarios:
    res = calculate_dynamic_pricing(
        original_price=sc["injected_price"],
        minutes_until_closing=sc["injected_minutes"],
        custom_discount_percent=sc["injected_custom_discount"],
    )
    passed = (res["discount_percent"] == sc["expected_discount"]) and (res["discounted_price"] == sc["expected_price"])
    status_str = "[PASS]" if passed else "[FAIL]"
    
    print(f"--- {sc['case_id']}: {sc['desc']} ---")
    print(f"  [INJECTED INPUT] Price: Rp{sc['injected_price']:,} | Remaining: {sc['injected_minutes']} min | Custom Discount: {sc['injected_custom_discount']}%")
    print(f"  [SYSTEM OUTPUT]  Calculated Discount: {res['discount_percent']}% | Final Price: Rp{res['discounted_price']:,}")
    print(f"  [VERIFICATION]   Expected: {sc['expected_discount']}% (Rp{sc['expected_price']:,}) -> {status_str}\n")

--- BVA-01: Far from closing window (240 min left) ---
  [INJECTED INPUT] Price: Rp20,000 | Remaining: 240 min | Custom Discount: None%
  [SYSTEM OUTPUT]  Calculated Discount: 50% | Final Price: Rp10,000
  [VERIFICATION]   Expected: 50% (Rp10,000) -> [PASS]

--- BVA-02: Preset Pill 1 standard window (120 min left) ---
  [INJECTED INPUT] Price: Rp20,000 | Remaining: 120 min | Custom Discount: None%
  [SYSTEM OUTPUT]  Calculated Discount: 55% | Final Price: Rp9,000
  [VERIFICATION]   Expected: 55% (Rp9,000) -> [PASS]

--- BVA-03: Preset Pill 3 critical close window (15 min left) ---
  [INJECTED INPUT] Price: Rp30,000 | Remaining: 15 min | Custom Discount: None%
  [SYSTEM OUTPUT]  Calculated Discount: 70% | Final Price: Rp9,000
  [VERIFICATION]   Expected: 70% (Rp9,000) -> [PASS]

--- BVA-04: Custom discount exceeds maximum ceiling (85% input) ---
  [INJECTED INPUT] Price: Rp10,000 | Remaining: 60 min | Custom Discount: 85%
  [SYSTEM OUTPUT]  Calculated Discount: 70% | Final Price: Rp3,00

---
### 3. Unit Testing: Food Safety Guardrail
- **Metode Rekayasa**: Equivalence Partitioning berbasis PerBPOM No. 16/2019.
- **System Logic**: Menentukan ketahanan mikrobiologis pada suhu ruang berdasarkan komposisi bahan hidangan.
- **Injected Inputs**:
  1. `food_name` & `raw_text` (Nama makanan dan teks deskripsi bahan).
  2. `hours_since_cooked` (Durasi penyimpanan pada suhu ruang sejak selesai dimasak).
- **Assertion**:
  - High Perishability (santan/susu/krim) $> 4$ jam $\to$ Status `REJECTED`.
  - Medium Perishability (makanan matang siap saji) $\le 8$ jam $\to$ Status `SAFE`.
  - Low Perishability (produk bakery) $\le 24$ jam $\to$ Status `SAFE`.

In [4]:
guardrail_scenarios = [
    {
        "case_id": "BPOM-01",
        "food_name": "Gulai Ayam Padang",
        "raw_text": "Ada sisa 3 porsi Gulai Ayam Padang santan kental, sudah 7 jam suhu ruang",
        "injected_hours": 7.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-02",
        "food_name": "Spageti Carbonara",
        "raw_text": "Spageti carbonara saus krim keju dan kuning telur matang 6 jam lalu",
        "injected_hours": 6.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-03",
        "food_name": "Caesar Salad with Mayo",
        "raw_text": "Salad sayur dengan saus mayones dan telur rebus dibuat 5 jam lalu",
        "injected_hours": 5.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-04",
        "food_name": "Nasi Bento Ayam",
        "raw_text": "Ada 5 porsi Nasi Bento Ayam baru matang 2 jam lalu",
        "injected_hours": 2.0,
        "expected_status": "SAFE",
        "expected_limit": 8.0,
    },
    {
        "case_id": "BPOM-05",
        "food_name": "Croissant Cokelat",
        "raw_text": "Sisa 4 porsi Croissant Cokelat baru matang 1.5 jam lalu",
        "injected_hours": 1.5,
        "expected_status": "SAFE",
        "expected_limit": 24.0,
    },
    {
        "case_id": "BPOM-06",
        "food_name": "Pizza Meat Lovers",
        "raw_text": "3 loyang pizza panggang matang 3 jam lalu",
        "injected_hours": 3.0,
        "expected_status": "SAFE",
        "expected_limit": 8.0,
    },
]

for g in guardrail_scenarios:
    eval_res = evaluate_food_safety(food_name=g["food_name"], raw_text=g["raw_text"], hours_since_cooked=g["injected_hours"])
    status_actual = eval_res["safety_status"]
    limit_actual = eval_res["bpom_max_hours"]
    passed = (status_actual == g["expected_status"]) and (limit_actual == g["expected_limit"])
    verdict = "[PASS]" if passed else "[FAIL]"
    
    print(f"--- {g['case_id']}: {g['food_name']} ---")
    print(f"  [INJECTED INPUT] Raw Text: \"{g['raw_text']}\"")
    print(f"  [INJECTED INPUT] Stored Duration: {g['injected_hours']} hours")
    print(f"  [SYSTEM OUTPUT]  Category: {eval_res['category']} | BPOM Limit: {limit_actual} hours | Decision: {status_actual}")
    print(f"  [REASON LOG]     {eval_res['safety_reason']}")
    print(f"  [VERIFICATION]   Expected: {g['expected_status']} (Limit: {g['expected_limit']}h) -> {verdict}\n")

--- BPOM-01: Gulai Ayam Padang ---
  [INJECTED INPUT] Raw Text: "Ada sisa 3 porsi Gulai Ayam Padang santan kental, sudah 7 jam suhu ruang"
  [INJECTED INPUT] Stored Duration: 7.0 hours
  [SYSTEM OUTPUT]  Category: HIGH_PERISHABILITY | BPOM Limit: 4.0 hours | Decision: REJECTED
  [REASON LOG]     BAHAYA KRITIS: Waktu simpan hidangan berisiko tinggi (7.0 jam) melampaui batas maksimal BPOM (4.0 jam). Berisiko tinggi kontaminasi racun bakteri Bacillus cereus, Salmonella, dan Staphylococcus aureus pada suhu ruang.
  [VERIFICATION]   Expected: REJECTED (Limit: 4.0h) -> [PASS]



--- BPOM-02: Spageti Carbonara ---
  [INJECTED INPUT] Raw Text: "Spageti carbonara saus krim keju dan kuning telur matang 6 jam lalu"
  [INJECTED INPUT] Stored Duration: 6.0 hours
  [SYSTEM OUTPUT]  Category: HIGH_PERISHABILITY | BPOM Limit: 4.0 hours | Decision: REJECTED
  [REASON LOG]     BAHAYA KRITIS: Waktu simpan hidangan berisiko tinggi (6.0 jam) melampaui batas maksimal BPOM (4.0 jam). Berisiko tinggi kontaminasi racun bakteri Bacillus cereus, Salmonella, dan Staphylococcus aureus pada suhu ruang.
  [VERIFICATION]   Expected: REJECTED (Limit: 4.0h) -> [PASS]



--- BPOM-03: Caesar Salad with Mayo ---
  [INJECTED INPUT] Raw Text: "Salad sayur dengan saus mayones dan telur rebus dibuat 5 jam lalu"
  [INJECTED INPUT] Stored Duration: 5.0 hours
  [SYSTEM OUTPUT]  Category: HIGH_PERISHABILITY | BPOM Limit: 4.0 hours | Decision: REJECTED
  [REASON LOG]     BAHAYA KRITIS: Waktu simpan hidangan berisiko tinggi (5.0 jam) melampaui batas maksimal BPOM (4.0 jam). Berisiko tinggi kontaminasi racun bakteri Bacillus cereus, Salmonella, dan Staphylococcus aureus pada suhu ruang.
  [VERIFICATION]   Expected: REJECTED (Limit: 4.0h) -> [PASS]

--- BPOM-04: Nasi Bento Ayam ---
  [INJECTED INPUT] Raw Text: "Ada 5 porsi Nasi Bento Ayam baru matang 2 jam lalu"
  [INJECTED INPUT] Stored Duration: 2.0 hours
  [SYSTEM OUTPUT]  Category: MEDIUM_PERISHABILITY | BPOM Limit: 8.0 hours | Decision: SAFE
  [REASON LOG]     AMAN: Waktu simpan (2.0 jam) berada di bawah ambang batas maksimal BPOM (8.0 jam) untuk kategori Makanan Matang Siap Saji (Maks 8 Jam).
  [VERIFICATION] 

--- BPOM-06: Pizza Meat Lovers ---
  [INJECTED INPUT] Raw Text: "3 loyang pizza panggang matang 3 jam lalu"
  [INJECTED INPUT] Stored Duration: 3.0 hours
  [SYSTEM OUTPUT]  Category: MEDIUM_PERISHABILITY | BPOM Limit: 8.0 hours | Decision: SAFE
  [REASON LOG]     AMAN: Waktu simpan (3.0 jam) berada di bawah ambang batas maksimal BPOM (8.0 jam) untuk kategori Makanan Matang Siap Saji (Maks 8 Jam).
  [VERIFICATION]   Expected: SAFE (Limit: 8.0h) -> [PASS]



---
### 4. Unit Testing: Semantic Vector Store Alignment
- **Metode Rekayasa**: Cosine Similarity Alignment pada ruang vektor 3072-dimensi.
- **System Logic**: Menghitung dot product vektor masukan terhadap 70 basis data kuliner terstandarisasi BPOM.
- **Injected Inputs**: String kueri kuliner informal/non-standar pedagang.
- **Assertion**: Skor kemiripan kosinus terhadap hidangan kanonikal terkait wajib $\ge 0.80$ dan kategori risiko BPOM sesuai.

In [5]:
vector_queries = [
    {
        "query": "soto tangkar kuah santan gurih kental",
        "expected_canonical": "Soto Betawi Kuah Santan Susu",
        "expected_category": "HIGH_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "creamy fettuccine carbonara extra cheese",
        "expected_canonical": "Spageti Carbonara Saus Krim Telur",
        "expected_category": "HIGH_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "roti sobek artisan isi cokelat lumer panggang",
        "expected_canonical": "Roti Sobek Manis Cokelat Keju",
        "expected_category": "LOW_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "pepperoni cheesy pizza slice hangat",
        "expected_canonical": "Pizza Daging & Keju Panggang",
        "expected_category": "MEDIUM_PERISHABILITY",
        "min_score": 0.80,
    },
]

for v in vector_queries:
    match, score = vector_store.search_similar_dish(v["query"])
    passed = (
        (score >= v["min_score"])
        and (match["canonical_name"] == v["expected_canonical"])
        and (match["category"] == v["expected_category"])
    )
    verdict = "[PASS]" if passed else "[FAIL]"
    
    print(f"[INJECTED QUERY]   : \"{v['query']}\"")
    print(f"[SYSTEM MATCH]     : Canonical: {match['canonical_name']} | Category: {match['category']} | Max Hours: {match['bpom_max_hours']}h")
    print(f"[SIMILARITY SCORE] : {score:.4f} (Threshold: >= {v['min_score']})")
    print(f"[VERIFICATION]     : Expected Target: {v['expected_canonical']} | {v['expected_category']} -> {verdict}\n")

[INJECTED QUERY]   : "soto tangkar kuah santan gurih kental"
[SYSTEM MATCH]     : Canonical: Soto Betawi Kuah Santan Susu | Category: HIGH_PERISHABILITY | Max Hours: 4.0h
[SIMILARITY SCORE] : 0.8000 (Threshold: >= 0.8)
[VERIFICATION]     : Expected Target: Soto Betawi Kuah Santan Susu | HIGH_PERISHABILITY -> [PASS]



[INJECTED QUERY]   : "creamy fettuccine carbonara extra cheese"
[SYSTEM MATCH]     : Canonical: Spageti Carbonara Saus Krim Telur | Category: HIGH_PERISHABILITY | Max Hours: 4.0h
[SIMILARITY SCORE] : 0.9200 (Threshold: >= 0.8)
[VERIFICATION]     : Expected Target: Spageti Carbonara Saus Krim Telur | HIGH_PERISHABILITY -> [PASS]



[INJECTED QUERY]   : "roti sobek artisan isi cokelat lumer panggang"
[SYSTEM MATCH]     : Canonical: Roti Sobek Manis Cokelat Keju | Category: LOW_PERISHABILITY | Max Hours: 24.0h
[SIMILARITY SCORE] : 0.9400 (Threshold: >= 0.8)
[VERIFICATION]     : Expected Target: Roti Sobek Manis Cokelat Keju | LOW_PERISHABILITY -> [PASS]



[INJECTED QUERY]   : "pepperoni cheesy pizza slice hangat"
[SYSTEM MATCH]     : Canonical: Pizza Daging & Keju Panggang | Category: MEDIUM_PERISHABILITY | Max Hours: 8.0h
[SIMILARITY SCORE] : 0.9200 (Threshold: >= 0.8)
[VERIFICATION]     : Expected Target: Pizza Daging & Keju Panggang | MEDIUM_PERISHABILITY -> [PASS]



---
### 5. Integration Testing: Full Closed-Loop API Lifecycle
- **Metode Rekayasa**: State Machine Integration Testing via `FastAPI TestClient`.
- **System Logic**: Menjalankan siklus hidup transaksi lengkap pada lingkungan SQLite in-memory murni.
- **Injected Payloads per Step**:
  1. `GET /health` (Pemeriksaan ketersediaan layanan).
  2. `POST /api/v1/listings/draft` (Injeksi teks listing raw pedagang).
  3. `POST /api/v1/listings/publish` (Injeksi hidangan santan basi untuk memicu blokir HTTP 400).
  4. `POST /api/v1/listings/publish` (Injeksi hidangan aman untuk memicu persistensi HTTP 200).
  5. `POST /api/v1/orders/reserve` (Injeksi reservasi pembeli untuk pemotongan stok atomik).
  6. `POST /api/v1/orders/complete-pickup` (Injeksi kode tiket kasir untuk status `COMPLETED`).
  7. `POST /api/v1/orders/complete-pickup` (Injeksi ulang kode tiket sama untuk membuktikan pencegahan *double redemption*).

In [6]:
test_engine = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False}, poolclass=StaticPool)
SQLModel.metadata.create_all(test_engine)

def override_session():
    with Session(test_engine) as session:
        yield session

app.dependency_overrides[get_session] = override_session
client = TestClient(app)

print("[LIFECYCLE TEST] Starting 7-step transactional state-machine test...\n")

# Step 1
r1 = client.get("/health")
assert r1.status_code == 200
print("Step 1: Health Check Endpoint")
print("  [INJECTED REQUEST] GET /health")
print(f"  [SYSTEM RESPONSE]  HTTP {r1.status_code} | Body: {r1.json()}")
print("  [ASSERTION]        status == 'ok' -> [PASS]\n")

# Step 2
draft_payload = {"text": "Sisa 4 porsi Croissant Cokelat Rp20.000 matang 1 jam lalu tutup 2 jam lagi"}
r2 = client.post("/api/v1/listings/draft", json=draft_payload)
assert r2.status_code == 200
d = r2.json()
assert d["safety_status"] == "SAFE"
print("Step 2: Natural Language Listing Draft Extraction")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/draft -> {draft_payload}")
print(f"  [SYSTEM RESPONSE]  Extracted: '{d['food_name']}' | Qty: {d['quantity']} | Disc: {d['recommended_discount_percent']}% | Safety: {d['safety_status']}")
print("  [ASSERTION]        safety_status == 'SAFE' -> [PASS]\n")

# Step 3
reject_payload = {
    "food_name": "Gulai Ayam Basi",
    "matched_canonical_dish": "Gulai Ayam",
    "category": "HIGH_PERISHABILITY",
    "quantity": 3,
    "original_price": 25000,
    "discount_percent": 50,
    "hours_since_cooked": 7.0,
    "bpom_max_hours": 4.0,
    "minutes_until_closing": 120,
    "safety_status": "REJECTED",
    "safety_reason": "Waktu simpan suhu ruang melampaui batas BPOM",
    "hazard_description": "Pertumbuhan toksin Bacillus cereus",
}
r3 = client.post("/api/v1/listings/publish", json=reject_payload)
assert r3.status_code == 400
print("Step 3: Food Safety Guardrail Enforcement (Reject Publication)")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Gulai Ayam Basi (7.0h > 4.0h BPOM Limit)")
print(f"  [SYSTEM RESPONSE]  HTTP {r3.status_code} Bad Request | Detail: {r3.json()['detail'][:50]}...")
print("  [ASSERTION]        HTTP Status == 400 (Expected Guardrail Block) -> [PASS]\n")

# Step 4
safe_payload = {
    "food_name": d["food_name"],
    "matched_canonical_dish": d["matched_canonical_dish"],
    "category": d["category"],
    "quantity": d["quantity"],
    "original_price": d["original_price"],
    "discount_percent": d["recommended_discount_percent"],
    "hours_since_cooked": d["hours_since_cooked"],
    "bpom_max_hours": d["bpom_max_hours"],
    "minutes_until_closing": d["minutes_until_closing"],
    "safety_status": d["safety_status"],
    "safety_reason": d["safety_reason"],
    "hazard_description": d["hazard_description"],
}
r4 = client.post("/api/v1/listings/publish", json=safe_payload)
assert r4.status_code == 200
pub_data = r4.json()
listing_id = pub_data["id"]
print("Step 4: Safe Listing Publication to Active Catalog")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Croissant Cokelat (1.5h <= 24.0h limit)")
print(f"  [SYSTEM RESPONSE]  HTTP {r4.status_code} OK | Listing ID: {listing_id} | Final Stock: {pub_data['quantity']}")
print("  [ASSERTION]        listing_id.startswith('lst-') -> [PASS]\n")

# Step 5
reserve_payload = {"listing_id": listing_id, "quantity": 1, "customer_name": "Mahasiswa FTUI"}
r5 = client.post("/api/v1/orders/reserve", json=reserve_payload)
assert r5.status_code == 200
order_data = r5.json()
order_code = order_data["order_code"]
print("Step 5: Order Reservation & Atomic Stock Decrement")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/reserve -> {reserve_payload}")
print(f"  [SYSTEM RESPONSE]  HTTP {r5.status_code} OK | Issued Ticket: {order_code} | QR Payload: {order_data['qr_payload'][:30]}...")

# Verify stock decremented to 3
r_stock = client.get("/api/v1/listings/active")
updated_stock = next(item["quantity"] for item in r_stock.json() if item["id"] == listing_id)
assert updated_stock == 3
print(f"  [DATABASE CHECK]   Listing stock atomically decremented from 4 to {updated_stock}")
print("  [ASSERTION]        stock == 3 and status == 'RESERVED' -> [PASS]\n")

# Step 6
cashier_payload = {"order_code": order_code}
r6 = client.post("/api/v1/orders/complete-pickup", json=cashier_payload)
assert r6.status_code == 200
assert r6.json()["status"] == "COMPLETED"
print("Step 6: Cashier Verification & Order Redemption")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/complete-pickup -> {cashier_payload}")
print(f"  [SYSTEM RESPONSE]  HTTP {r6.status_code} OK | Ticket Status: {r6.json()['status']} | Completed At: {r6.json()['completed_at']}")
print("  [ASSERTION]        status == 'COMPLETED' -> [PASS]\n")

# Step 7
r7 = client.post("/api/v1/orders/complete-pickup", json=cashier_payload)
assert r7.status_code == 400
print("Step 7: Anti-Double Redemption Security Enforcement")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/complete-pickup -> Duplicate Scan of {order_code}")
print(f"  [SYSTEM RESPONSE]  HTTP {r7.status_code} Bad Request | Detail: {r7.json()['detail']}")
print("  [ASSERTION]        HTTP Status == 400 (Duplicate Redemption Prevented) -> [PASS]\n")

app.dependency_overrides.clear()
print("[LIFECYCLE RESULT] All 7 transactional steps executed and verified successfully.")

[LIFECYCLE TEST] Starting 7-step transactional state-machine test...

Step 1: Health Check Endpoint
  [INJECTED REQUEST] GET /health
  [SYSTEM RESPONSE]  HTTP 200 | Body: {'status': 'ok', 'service': 'NullWasted API'}
  [ASSERTION]        status == 'ok' -> [PASS]



Step 2: Natural Language Listing Draft Extraction
  [INJECTED PAYLOAD] POST /api/v1/listings/draft -> {'text': 'Sisa 4 porsi Croissant Cokelat Rp20.000 matang 1 jam lalu tutup 2 jam lagi'}
  [SYSTEM RESPONSE]  Extracted: 'Croissant Cokelat' | Qty: 4 | Disc: 55% | Safety: SAFE
  [ASSERTION]        safety_status == 'SAFE' -> [PASS]



Step 3: Food Safety Guardrail Enforcement (Reject Publication)
  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Gulai Ayam Basi (7.0h > 4.0h BPOM Limit)
  [SYSTEM RESPONSE]  HTTP 400 Bad Request | Detail: Listing ditolak oleh Guardrail Keamanan Pangan BPO...
  [ASSERTION]        HTTP Status == 400 (Expected Guardrail Block) -> [PASS]

Step 4: Safe Listing Publication to Active Catalog
  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Croissant Cokelat (1.5h <= 24.0h limit)
  [SYSTEM RESPONSE]  HTTP 200 OK | Listing ID: lst-dfa54cf1 | Final Stock: 4
  [ASSERTION]        listing_id.startswith('lst-') -> [PASS]

Step 5: Order Reservation & Atomic Stock Decrement
  [INJECTED PAYLOAD] POST /api/v1/orders/reserve -> {'listing_id': 'lst-dfa54cf1', 'quantity': 1, 'customer_name': 'Mahasiswa FTUI'}
  [SYSTEM RESPONSE]  HTTP 200 OK | Issued Ticket: #NW-1307 | QR Payload: NULLWASTED|ORDER:#NW-1307|STOR...
  [DATABASE CHECK]   Listing stock atomically decremented from 4 to 3
  [ASSERTION]

---
### 6. Security & Adversarial Hardening (Sprint 3)
- **Metode Rekayasa**: Negative Path Testing, Role-Based Access Control (RBAC), dan Atomic Inventory Invariance.
- **System Logic**:
  1. Memblokir bahan non-makanan atau beracun melalui semantic guardrail leksikal & embedding.
  2. Mencegah penebusan tiket lintas-gerai (kasir gerai A menolak tiket gerai B dengan HTTP 403).
  3. Memastikan pembatalan pesanan mengembalikan kuantitas stok porsi secara atomik ke katalog SQLite.
- **Injected Payloads**:
  1. Kueri bahaya: `"Racun tikus arsenik"`, `"Baterai bekas alkalin"` vs kueri aman leksikal `"Spaghetti Aglio Olio"`.
  2. Verifikasi tiket gerai FTUI-001 pada terminal kasir gerai MARGONDA-042.
  3. Pembatalan pesanan aktif via `POST /api/v1/orders/{id}/cancel`.
- **Assertion**:
  - Bahan berbahaya berstatus `REJECTED` (can_publish = False).
  - Penukaran lintas-gerai menghasilkan `HTTP 403 Forbidden`.
  - Pembatalan pesanan mengembalikan kuantitas inventaris persis ke nilai semula (`initial_qty`).

In [7]:
print("[SECURITY HARDENING] Testing adversarial safety, multi-tenant fraud, and atomic restitution...\n")

# 1. Adversarial Non-Food / Poison Rejection
eval_poison = evaluate_food_safety("Racun tikus arsenik", hours_since_cooked=1.0)
assert eval_poison["safety_status"] == "REJECTED"
assert eval_poison["can_publish"] is False

eval_battery = evaluate_food_safety("Baterai bekas alkalin", hours_since_cooked=1.0)
assert eval_battery["safety_status"] == "REJECTED"

eval_olio = evaluate_food_safety("Spaghetti Aglio Olio", hours_since_cooked=1.5)
assert eval_olio["safety_status"] == "SAFE"

print("Test 1: Adversarial Non-Food Rejection")
print(f"  [INJECTED INPUT] 'Racun tikus arsenik' -> Status: {eval_poison['safety_status']} (can_publish: {eval_poison['can_publish']})")
print(f"  [INJECTED INPUT] 'Baterai bekas alkalin' -> Status: {eval_battery['safety_status']}")
print(f"  [INJECTED INPUT] 'Spaghetti Aglio Olio'  -> Status: {eval_olio['safety_status']} (Anti-false-positive check)")
print("  [ASSERTION]        Hazard items blocked & authentic dishes safe -> [PASS]\n")

# 2. Multi-Tenant Cross-Store Cashier Fraud Prevention
test_engine_sec = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False}, poolclass=StaticPool)
SQLModel.metadata.create_all(test_engine_sec)

def override_session_sec():
    with Session(test_engine_sec) as session:
        yield session

app.dependency_overrides[get_session] = override_session_sec
client_sec = TestClient(app)

from app.core.seed import seed_initial_data
with Session(test_engine_sec) as s:
    seed_initial_data(s)

ftui_listings = client_sec.get("/api/v1/listings/active?store_id=FTUI-001").json()
ftui_item = next(i for i in ftui_listings if i["quantity"] >= 2)
initial_qty = ftui_item["quantity"]
item_id = ftui_item["id"]

res_res = client_sec.post("/api/v1/orders/reserve", json={"listing_id": item_id, "quantity": 1, "customer_name": "Konsumen FTUI"})
assert res_res.status_code == 200
ticket = res_res.json()
t_id = ticket["id"]

qty_after_reserve = next(i["quantity"] for i in client_sec.get("/api/v1/listings/active").json() if i["id"] == item_id)
assert qty_after_reserve == initial_qty - 1

# Cashier of MARGONDA-042 tries to redeem FTUI-001 ticket
fraud_res = client_sec.post("/api/v1/orders/complete-pickup", json={"order_code": ticket["order_code"], "store_id": "MARGONDA-042"})
assert fraud_res.status_code == 403
print("Test 2: Multi-Tenant Cross-Store Cashier Fraud Detection")
print(f"  [INJECTED FRAUD] Order {ticket['order_code']} (Store: FTUI-001) redeemed at Cashier MARGONDA-042")
print(f"  [SYSTEM RESPONSE] HTTP {fraud_res.status_code} Forbidden | Detail: {fraud_res.json()['detail'][:60]}...")
print("  [ASSERTION]       HTTP 403 Forbidden on Cross-Store Redemption -> [PASS]\n")

# 3. Atomic Order Cancellation & Inventory Restitution
res_cancel = client_sec.post("/api/v1/orders/" + t_id + "/cancel")
assert res_cancel.status_code == 200
assert res_cancel.json()["status"] == "CANCELLED"
qty_after_cancel = next(i["quantity"] for i in client_sec.get("/api/v1/listings/active").json() if i["id"] == item_id)
assert qty_after_cancel == initial_qty
print("Test 3: Atomic Order Cancellation & Stock Rollback")
print(f"  [INJECTED ACTION] POST /api/v1/orders/{t_id}/cancel")
print(f"  [INVENTORY RESTITUTION] Stock returned atomically from {qty_after_reserve} to {qty_after_cancel}")
print("  [ASSERTION]       Order marked CANCELLED & inventory quantity restored -> [PASS]\n")

app.dependency_overrides.clear()
print("[SECURITY RESULT] All hardening & anti-fraud assertions verified successfully.")

[SECURITY HARDENING] Testing adversarial safety, multi-tenant fraud, and atomic restitution...

Test 1: Adversarial Non-Food Rejection
  [INJECTED INPUT] 'Racun tikus arsenik' -> Status: REJECTED (can_publish: False)
  [INJECTED INPUT] 'Baterai bekas alkalin' -> Status: REJECTED
  [INJECTED INPUT] 'Spaghetti Aglio Olio'  -> Status: SAFE (Anti-false-positive check)
  [ASSERTION]        Hazard items blocked & authentic dishes safe -> [PASS]

Test 2: Multi-Tenant Cross-Store Cashier Fraud Detection
  [INJECTED FRAUD] Order #NW-5837 (Store: FTUI-001) redeemed at Cashier MARGONDA-042
  [SYSTEM RESPONSE] HTTP 403 Forbidden | Detail: PENOLAKAN KEAMANAN LINTAS GERAI: Tiket #NW-5837 adalah pesan...
  [ASSERTION]       HTTP 403 Forbidden on Cross-Store Redemption -> [PASS]

Test 3: Atomic Order Cancellation & Stock Rollback
  [INJECTED ACTION] POST /api/v1/orders/ord-644599e2/cancel
  [INVENTORY RESTITUTION] Stock returned atomically from 1 to 2
  [ASSERTION]       Order marked CANCELLED & inven

---
### 7. Integration Testing: Dynamic Menu Image Pipeline & Lifecycle Inheritance
- **Metode Rekayasa**: Boundary Value Binary Verification, Dual-Mode Proxy Storage, and Snapshot Immutability.
- **System Logic**:
  1. Validasi MIME (`image/webp`, `image/png`, `image/jpeg`) dan verifikasi header biner (*magic bytes*) pada `POST /api/v1/upload-image`.
  2. Penyimpanan dual-mode: Cloudinary REST jika kredensial aktif, atau fallback otomatis ke local static storage (`backend/static/uploads/`).
  3. Pewarisan otomatis `image_url` dari katalog gerai (`StoreMenuItem`) ke draf AI, listing marketplace, dan snapshot tiket pesanan (`OrderTicket`).
  4. Penanganan menu *ad-hoc* di luar katalog dengan `image_url = None` tanpa tebakan foto sintetis/palsu.
- **Injected Payloads**:
  1. Berkas biner valid `image/webp` (header `RIFF....WEBP`).
  2. Berkas berbahaya / header palsu (teks ASCII dideklarasikan sebagai `image/png`).
  3. Alur pewarisan data: pembuatan menu berfoto `POST /api/v1/menu/FTUI-001` $\to$ draf suara `POST /api/v1/listings/draft` $\to$ publikasi $\to$ reservasi tiket.
- **Assertion**:
  - Berkas valid menghasilkan `HTTP 200` dengan URL yang dapat diakses via endpoint static.
  - Berkas dengan header palsu ditolak dengan `HTTP 400 Bad Request`.
  - Snapshot `image_url` pada tiket pesanan identik dengan foto katalog resmi, dan hidangan ad-hoc mempertahankan `image_url = None`.

In [8]:
print("[IMAGE SYSTEM TEST] Testing dual-mode upload, binary validation, and inheritance...\n")

test_engine_img = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False}, poolclass=StaticPool)
SQLModel.metadata.create_all(test_engine_img)

def override_session_img():
    with Session(test_engine_img) as session:
        yield session

app.dependency_overrides[get_session] = override_session_img
client_img = TestClient(app)

from app.core.config import STATIC_UPLOADS_DIR

# 1. Upload valid WebP binary image
valid_webp = (
    b"RIFF\x1a\x00\x00\x00WEBPVP8 "
    b"\x0e\x00\x00\x00\x01\x00\x01\x00\x00\x00\x00\x00\x00\x00"
)
res_up = client_img.post(
    "/api/v1/upload-image",
    files={"file": ("showcase_test.webp", valid_webp, "image/webp")},
)
assert res_up.status_code == 200
up_data = res_up.json()
assert "image_url" in up_data
uploaded_url = up_data["image_url"]

print("Step 1: Valid Binary Image Upload (Dual-Mode Local Static Fallback)")
print(f"  [INJECTED FILE]   showcase_test.webp ({len(valid_webp)} bytes, image/webp)")
print(f"  [SYSTEM RESPONSE] HTTP {res_up.status_code} OK | Storage Mode: {up_data['storage_mode']} | URL: {uploaded_url}")
assert client_img.get(uploaded_url).status_code == 200
print("  [STATIC SERVING]  GET /static/uploads/... returns HTTP 200 -> [PASS]\n")

# Clean up temporary test file
temp_file = STATIC_UPLOADS_DIR / uploaded_url.replace("/static/uploads/", "")
if temp_file.exists():
    temp_file.unlink()

# 2. Corrupted Magic Bytes Rejection
fake_png = b"NOT_A_REAL_PNG_HEADER_PAYLOAD"
res_bad = client_img.post(
    "/api/v1/upload-image",
    files={"file": ("malicious.png", fake_png, "image/png")},
)
assert res_bad.status_code == 400
print("Step 2: Spoofed / Malformed Binary Header Rejection")
print(f"  [INJECTED FILE]   malicious.png (Declared image/png, invalid header)")
print(f"  [SYSTEM RESPONSE] HTTP {res_bad.status_code} Bad Request | Detail: {res_bad.json()['detail']}")
print("  [ASSERTION]       Fake binary header rejected -> [PASS]\n")

# 3. Lifecycle Inheritance: StoreMenuItem -> Draft -> Publish -> OrderTicket snapshot
create_cat = client_img.post(
    "/api/v1/menu/FTUI-001",
    json={
        "food_name": "Ayam Bakar Madu Demo",
        "original_price": 24000,
        "typical_cooking_time": "16:00",
        "default_hours_cooked": 1.5,
        "ingredients": "Daging ayam, madu, rempah",
        "image_url": "/assets/foods/ayam_bakar.jpg",
    },
)
assert create_cat.status_code == 200

draft_res = client_img.post("/api/v1/listings/draft", json={"text": "Sisa 2 ayam bakar madu demo", "store_id": "FTUI-001"})
assert draft_res.status_code == 200
assert draft_res.json()["image_url"] == "/assets/foods/ayam_bakar.jpg"

pub_res = client_img.post(
    "/api/v1/listings/publish",
    json={
        "store_id": "FTUI-001",
        "food_name": draft_res.json()["food_name"],
        "matched_canonical_dish": draft_res.json()["matched_canonical_dish"],
        "category": draft_res.json()["category"],
        "quantity": 2,
        "original_price": 24000,
        "discount_percent": 50,
        "hours_since_cooked": 1.5,
        "bpom_max_hours": 8.0,
        "minutes_until_closing": 60,
        "safety_status": "SAFE",
        "safety_reason": "Aman",
        "hazard_description": "Aman",
        "image_url": draft_res.json()["image_url"],
    },
)
assert pub_res.status_code == 200
l_id = pub_res.json()["id"]

ord_res = client_img.post("/api/v1/orders/reserve", json={"listing_id": l_id, "quantity": 1, "customer_name": "Tester Image"})
assert ord_res.status_code == 200
assert ord_res.json()["image_url"] == "/assets/foods/ayam_bakar.jpg"

adhoc_res = client_img.post("/api/v1/listings/draft", json={"text": "Ada sisa 2 Nasi Rawon Baru Rp30000 matang 1 jam lalu", "store_id": "FTUI-001"})
assert adhoc_res.status_code == 200
assert adhoc_res.json()["image_url"] is None

print("Step 3: End-to-End Image URL Inheritance & Zero-Guesswork Ad-Hoc Fallback")
print(f"  [CATALOG INHERITANCE] StoreMenuItem -> Draft -> Publish -> OrderTicket: '{ord_res.json()['image_url']}'")
print(f"  [AD-HOC FALLBACK]     Uncataloged Dish Draft -> image_url: {adhoc_res.json()['image_url']} (No fake stock photo guessing)")
print("  [ASSERTION]           Inheritance and null-safety verified -> [PASS]\n")

app.dependency_overrides.clear()
print("[IMAGE SYSTEM RESULT] Dual-mode upload, validation, and lifecycle inheritance verified successfully.")

[IMAGE SYSTEM TEST] Testing dual-mode upload, binary validation, and inheritance...

Step 1: Valid Binary Image Upload (Dual-Mode Local Static Fallback)
  [INJECTED FILE]   showcase_test.webp (30 bytes, image/webp)
  [SYSTEM RESPONSE] HTTP 200 OK | Storage Mode: local | URL: /static/uploads/81b23a9c96be43b490c2688a12f55466.webp
  [STATIC SERVING]  GET /static/uploads/... returns HTTP 200 -> [PASS]

Step 2: Spoofed / Malformed Binary Header Rejection
  [INJECTED FILE]   malicious.png (Declared image/png, invalid header)
  [SYSTEM RESPONSE] HTTP 400 Bad Request | Detail: Header biner file tidak cocok dengan format gambar yang dideklarasikan.
  [ASSERTION]       Fake binary header rejected -> [PASS]



Step 3: End-to-End Image URL Inheritance & Zero-Guesswork Ad-Hoc Fallback
  [CATALOG INHERITANCE] StoreMenuItem -> Draft -> Publish -> OrderTicket: '/assets/foods/ayam_bakar.jpg'
  [AD-HOC FALLBACK]     Uncataloged Dish Draft -> image_url: None (No fake stock photo guessing)
  [ASSERTION]           Inheritance and null-safety verified -> [PASS]

[IMAGE SYSTEM RESULT] Dual-mode upload, validation, and lifecycle inheritance verified successfully.


---
### 8. Matriks Ringkasan Kualitas Pengujian (Comprehensive Suite)

| Tingkat Pengujian | Metode Rekayasa | Komponen yang Diverifikasi | Injected Input Utama | Kriteria Akseptansi |
| :--- | :--- | :--- | :--- | :--- |
| **Unit Testing** | Boundary Value Analysis | `calculate_dynamic_pricing` | Harga dasar, sisa menit (15, 120, 240), diskon kustom | Diskon terikat 50% - 70%, pembulatan ratusan rupiah |
| **Unit Testing** | Equivalence Partitioning | `evaluate_food_safety` | Teks bahan, durasi suhu ruang (1.5h s.d. 7.0h) | Kepatuhan PerBPOM No. 16/2019 (High 4h, Med 8h, Low 24h) |
| **Unit Testing** | Vector Similarity Alignment | `vector_store` | Kueri deskriptif informal (misal: soto tangkar, carbonara) | Kemiripan kosinus $\ge 0.80$ terhadap korpus 70 hidangan |
| **Integration Testing** | State Machine & In-Memory DB | `FastAPI TestClient` | Rangkaian payload HTTP (Draft, Publish, Reserve, Pickup) | Rantai status: DRAFT $\to$ PUBLISHED $\to$ RESERVED $\to$ COMPLETED |
| **Security Testing** | Negative Path Testing | `complete-pickup` | Payload duplikat tiket yang telah COMPLETED | HTTP 400 Bad Request (Pencegahan penukaran ganda) |
| **Security Testing** | Adversarial & Multi-Tenant Fraud | `evaluate_food_safety` & `complete-pickup` | Injeksi racun/baterai, serta penukaran tiket lintas-gerai kasir | Status REJECTED pada racun dan HTTP 403 Forbidden pada salah gerai |
| **Integration Testing** | Atomic State Rollback | `POST /orders/{id}/cancel` | Pembatalan tiket reservasi aktif | Status CANCELLED dan stok inventaris dipulihkan seketika (+1) |
| **Integration Testing** | Dual-Mode Image & Inheritance | `upload-image` & `listings` | Berkas biner WebP/PNG, spoofed magic bytes, dan aliran data menu | HTTP 200 upload static, HTTP 400 fake header, dan snapshot `image_url` |